Just a dev notebook to develop the algorithm. Will export the finals into a py file for execution. 

In [22]:
import os 
import sys

# set root folder to project root
root_path = os.path.abspath(os.path.join(".."))
if root_path not in sys.path:
    sys.path.insert(0, root_path)

# auto reload
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
import pandas as pd
import numpy as np
from pprint import pprint
from modules.bin_calcs import (
    PositionAngles,
    create_rotation_matrices, 
    compute_incremental_rotation_matrices, 
    decompose_rotation_matrices_yxy, 
    get_position_angles, 
    normalize_position_angles,
    extract_bin_data
)
from modules.data_loading import Heatmap
from modules.data_loading import load_participant_details, load_motion_capture_data

participant_details = load_participant_details('../data/raw_normalized_data/participant_details.xlsx')
participant = participant_details[0]
data = load_motion_capture_data(participant.filename)

In [24]:
# create R matrices
rotation_matrices = create_rotation_matrices(data, "left")

In [25]:
# determine postural position
postural_angles = get_position_angles(rotation_matrices)
postural_angles_normalized = normalize_position_angles(postural_angles)

In [26]:
# compute relative motion between each frame
relative_matrices = compute_incremental_rotation_matrices(rotation_matrices)

In [27]:
# decompose relative motion matrices into euler angles
euler_angles = decompose_rotation_matrices_yxy(relative_matrices)

c:\Users\chris\anaconda3\envs\rtsa_mocap\Lib\site-packages\IPython\core\interactiveshell.py:3748: UserWarning: Gimbal lock detected. Setting third angle to zero since it is not possible to uniquely determine all angles.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [29]:
# calculate cumulative motion in each axis
side_object = getattr(participant, "left")  # or "right" depending on the side
data_object: Heatmap = side_object.humerothoracic.heatmap
# for each bin
for elevation_range_start in range(0, 180, int(data_object.bin_width)):

    for poe_range_start in range(0, 360, int(data_object.bin_width)):
        # extract bin boundaries
        elevation_start = elevation_range_start
        elevation_end = elevation_range_start + data_object.bin_width
        poe_start = poe_range_start
        poe_end = poe_range_start + data_object.bin_width
        print(f"Processing bin: Elevation {elevation_start}-{elevation_end}, POE {poe_start}-{poe_end}")

        # filter extract only data within the bin boundaries
        bin_data = extract_bin_data(
            mocap_data=relative_matrices, 
            postural_data=postural_angles_normalized, 
            elevation_start=elevation_start, 
            elevation_end=elevation_end, 
            poe_start=poe_start, 
            poe_end=poe_end
        )
        # create bin mask based ont he absolute position angles
            # base it on start position
        # use the mask to return only the data within the bin boundaries
    # sum motion in each axis for the bin
    # save the bin calcs to the data object

Processing bin: Elevation 0-20, POE 0-20


IndexError: boolean index did not match indexed array along dimension 0; dimension is 405460 but corresponding boolean dimension is 405461